# BatteryIQ — Phase 2: XGBoost RUL Prediction

**Member 1 | Branch:** `feature/rul-xgboost`

---

## Objective
Predict the **Remaining Useful Life (RUL)** of lithium-ion batteries using cycle-level features and an XGBoost regression model.

## Pipeline
1. RUL Target Construction (SOH → EOL → RUL)
2. Dataset Validation
3. Feature / Target Merge
4. Leave-One-Battery-Out (LOGO) Cross-Validation
5. XGBoost Training & Evaluation (MAE, RMSE, R²)

## EOL Definition
Battery reaches **End-of-Life (EOL)** at the first cycle where `SOH ≤ 70%`.

```
RUL = EOL_cycle − current_cycle
```

## B0007 Note
> B0007 never reaches 70% SOH in this dataset. It is treated as **censored**: all rows have `rul_valid = False` and are **excluded from training and evaluation**.

---
## 0. Imports & Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Suppress OpenBLAS threading warnings
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# Resolve project root (works from notebooks/ directory)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("Python       :", sys.version.split()[0])

In [ ]:
# File paths
TARGETS_FILE  = PROJECT_ROOT / "data" / "features" / "battery_cycle_targets.csv"
FEATURES_FILE = PROJECT_ROOT / "data" / "features" / "battery_cycle_features.csv"
RUL_FILE      = PROJECT_ROOT / "data" / "features" / "rul_targets.csv"

for f in [TARGETS_FILE, FEATURES_FILE, RUL_FILE]:
    status = "✅" if f.exists() else "❌ MISSING"
    print(f"{status}  {f.name}")

---
## 1. RUL Target Construction

Load the frozen `battery_cycle_targets.csv` and compute RUL for each battery-cycle pair.

**Logic:**
- EOL = first cycle where SOH ≤ 70%
- RUL = EOL_cycle − current_cycle  (0 at EOL, None beyond EOL)
- B0007 never reaches EOL → all rows marked censored (`rul_valid = False`)

In [ ]:
# Load Phase 1 frozen target dataset
targets_df = pd.read_csv(TARGETS_FILE)

print("Shape:", targets_df.shape)
print("Columns:", targets_df.columns.tolist())
print()
targets_df.head(8)

In [ ]:
# Check SOH range per battery
targets_df.groupby("battery_id")["soh"].agg(["min", "max", "count"]).round(2)

In [ ]:
def construct_rul_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construct RUL targets from SOH data.

    EOL = first cycle where SOH <= 70%
    RUL = EOL_cycle - current_cycle   (rul_valid=True for cycles <= EOL)
    Censored batteries (never reach EOL) → rul_valid=False, rul=None
    """
    result_rows = []

    for battery_id, group in df.groupby("battery_id"):
        group = group.sort_values("cycle").reset_index(drop=True)
        eol_records = group[group["soh"] <= 70]

        if eol_records.empty:
            # Censored — B0007
            print(f"  {battery_id}: EOL not observed → censored")
            for _, row in group.iterrows():
                result_rows.append(dict(battery_id=battery_id,
                                        cycle=int(row["cycle"]),
                                        rul=None, rul_valid=False))
        else:
            eol_cycle = int(eol_records["cycle"].iloc[0])
            print(f"  {battery_id}: EOL at cycle {eol_cycle}")
            for _, row in group.iterrows():
                c = int(row["cycle"])
                if c <= eol_cycle:
                    result_rows.append(dict(battery_id=battery_id,
                                            cycle=c,
                                            rul=eol_cycle - c,
                                            rul_valid=True))
                else:
                    result_rows.append(dict(battery_id=battery_id,
                                            cycle=c,
                                            rul=None, rul_valid=False))

    return pd.DataFrame(result_rows)


print("Constructing RUL targets...")
rul_df = construct_rul_targets(targets_df)
print(f"\nTotal rows : {len(rul_df)}")
print(f"Valid rows : {rul_df['rul_valid'].sum()}")
rul_df.head(8)

In [ ]:
# Plot: SOH and RUL curves per battery
batteries = sorted(targets_df["battery_id"].unique())
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
colors = {"B0005": "#4C72B0", "B0006": "#DD8452", "B0007": "#55A868", "B0018": "#C44E52"}

for i, bid in enumerate(batteries):
    t = targets_df[targets_df["battery_id"] == bid].sort_values("cycle")
    r = rul_df[(rul_df["battery_id"] == bid) & (rul_df["rul_valid"] == True)]

    # SOH
    ax = axes[0, i]
    ax.plot(t["cycle"], t["soh"], color=colors[bid], linewidth=2)
    ax.axhline(70, color="red", linestyle="--", linewidth=1, label="EOL threshold")
    ax.set_title(f"{bid} — SOH", fontweight="bold")
    ax.set_xlabel("Cycle")
    ax.set_ylabel("SOH (%)")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # RUL
    ax2 = axes[1, i]
    if r.empty:
        ax2.text(0.5, 0.5, "Censored\n(B0007)", ha="center", va="center",
                 transform=ax2.transAxes, fontsize=12, color="gray")
    else:
        ax2.plot(r["cycle"], r["rul"], color=colors[bid], linewidth=2)
        ax2.set_xlabel("Cycle")
        ax2.set_ylabel("RUL (cycles)")
    ax2.set_title(f"{bid} — RUL", fontweight="bold")
    ax2.grid(alpha=0.3)

plt.suptitle("SOH Degradation and RUL per Battery", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Dataset Validation

Verify:
- ✅ No negative RUL
- ✅ RUL = 0 at EOL for each valid battery
- ✅ RUL decreases monotonically with cycle
- ✅ One row per battery-cycle (no duplicates)
- ✅ B0007 explicitly marked censored

In [ ]:
def validate_rul(rul_df: pd.DataFrame) -> None:
    checks = {}

    # 1. Required columns
    required = {"battery_id", "cycle", "rul", "rul_valid"}
    checks["Required columns present"] = required.issubset(rul_df.columns)

    # 2. No duplicates
    checks["No duplicate battery_id+cycle"] = not rul_df.duplicated(["battery_id", "cycle"]).any()

    valid = rul_df[rul_df["rul_valid"] == True].copy()

    # 3. No negative RUL
    checks["No negative RUL"] = (valid["rul"] >= 0).all()

    # 4. RUL = 0 at EOL
    eol_check = all(
        (valid[valid["battery_id"] == bid]["rul"] == 0).any()
        for bid in valid["battery_id"].unique()
    )
    checks["RUL = 0 at EOL for valid batteries"] = eol_check

    # 5. Monotone decrease
    mono_check = all(
        valid[valid["battery_id"] == bid].sort_values("cycle")["rul"].is_monotonic_decreasing
        for bid in valid["battery_id"].unique()
    )
    checks["RUL decreases with cycle"] = mono_check

    # 6. B0007 censored
    b0007 = rul_df[rul_df["battery_id"] == "B0007"]
    checks["B0007 fully censored"] = (not b0007.empty) and (not b0007["rul_valid"].any())

    # Print results
    print("=" * 45)
    print("  RUL Validation Report")
    print("=" * 45)
    all_passed = True
    for check, passed in checks.items():
        icon = "✅" if passed else "❌"
        print(f"  {icon}  {check}")
        if not passed:
            all_passed = False
    print("=" * 45)
    if all_passed:
        print("  All checks passed.")
    else:
        raise AssertionError("One or more validation checks failed!")

validate_rul(rul_df)

In [ ]:
# Summary table
summary = rul_df.groupby(["battery_id", "rul_valid"]).agg(
    rows=("cycle", "count"),
    rul_min=("rul", "min"),
    rul_max=("rul", "max")
).reset_index()

print("RUL dataset summary:")
print(summary.to_string(index=False))

---
## 3. Feature / Target Merge

Merge the frozen Phase 1 features with RUL targets on `[battery_id, cycle]`.
Keep only valid RUL rows (`rul_valid = True`).

In [ ]:
features_df = pd.read_csv(FEATURES_FILE)

print("Feature dataset shape :", features_df.shape)
print("Columns:", features_df.columns.tolist())

In [ ]:
# Merge features + RUL targets
merged_df = features_df.merge(
    rul_df[["battery_id", "cycle", "rul", "rul_valid"]],
    on=["battery_id", "cycle"],
    how="inner"
)

# Keep only valid RUL rows (exclude censored B0007 + post-EOL rows)
model_df = merged_df[merged_df["rul_valid"] == True].copy()

print(f"After merge          : {merged_df.shape}")
print(f"Valid RUL rows only  : {model_df.shape}")
print(f"Batteries in model   : {sorted(model_df['battery_id'].unique())}")
model_df[["battery_id", "cycle", "rul"]].head(8)

In [ ]:
# Feature columns from frozen Phase 1
FEATURE_COLUMNS = [
    "voltage_mean", "voltage_min", "voltage_max", "voltage_std",
    "current_mean", "current_min", "current_max",
    "temperature_mean", "temperature_max",
    "current_load_mean", "voltage_load_mean",
    "cycle_time", "voltage_change", "temperature_change",
    "temperature_rolling_mean",
]

missing_cols = [c for c in FEATURE_COLUMNS if c not in model_df.columns]
if missing_cols:
    raise ValueError(f"Missing feature columns: {missing_cols}")

X      = model_df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
y      = model_df["rul"].astype(float)
groups = model_df["battery_id"]

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"NaN in X: {X.isnull().sum().sum()}")
print(f"RUL range: [{y.min():.0f}, {y.max():.0f}]")

---
## 4. Leave-One-Battery-Out (LOGO) Cross-Validation

Battery-level splits prevent data leakage.  
Each fold trains on 2 batteries and tests on 1.

| Fold | Train batteries | Test battery |
|:----:|:---------------:|:------------:|
| 1 | B0006, B0018 | B0005 |
| 2 | B0005, B0018 | B0006 |
| 3 | B0005, B0006 | B0018 |

In [ ]:
logo = LeaveOneGroupOut()

n_splits = logo.get_n_splits(X, y, groups=groups)
print(f"Number of LOGO folds: {n_splits}")

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups=groups), start=1
):
    train_bats = sorted(groups.iloc[train_idx].unique())
    test_bat   = groups.iloc[test_idx].unique()[0]
    print(f"  Fold {fold}: train={train_bats}  test={test_bat}")

---
## 5. XGBoost Training & Evaluation

In [ ]:
all_actual:      list = []
all_predictions: list = []
fold_results:    list = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups=groups), start=1
):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    test_battery = groups.iloc[test_idx].unique()[0]

    print(f"\nFold {fold} — Test battery: {test_battery}")
    print(f"  Train size: {len(X_train)}  |  Test size: {len(X_test)}")

    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=1,
        verbosity=0,
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    all_actual.extend(y_test.tolist())
    all_predictions.extend(preds.tolist())

    mae  = mean_absolute_error(y_test, preds)
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    r2   = r2_score(y_test, preds)

    fold_results.append(dict(fold=fold, test_battery=test_battery,
                              n_test=len(y_test),
                              mae=round(mae, 4),
                              rmse=round(rmse, 4),
                              r2=round(r2, 4)))

    print(f"  MAE : {mae:.4f} cycles")
    print(f"  RMSE: {rmse:.4f} cycles")
    print(f"  R²  : {r2:.4f}")

---
## 6. Results — MAE, RMSE, R²

In [ ]:
# Overall metrics (pooled)
overall_mae  = mean_absolute_error(all_actual, all_predictions)
overall_rmse = float(np.sqrt(mean_squared_error(all_actual, all_predictions)))
overall_r2   = r2_score(all_actual, all_predictions)

print("=" * 50)
print("  OVERALL XGBoost RUL Results (LOGO)")
print("=" * 50)
print(f"  MAE  (cycles) : {overall_mae:.4f}")
print(f"  RMSE (cycles) : {overall_rmse:.4f}")
print(f"  R²            : {overall_r2:.4f}")
print("=" * 50)

results_df = pd.DataFrame(fold_results)
print("\nPer-fold results:")
print(results_df.to_string(index=False))

In [ ]:
# Plot 1: Actual vs Predicted per fold
fig, axes = plt.subplots(1, len(fold_results), figsize=(5 * len(fold_results), 4), sharey=False)
if len(fold_results) == 1:
    axes = [axes]

fold_colors = ["#4C72B0", "#DD8452", "#C44E52"]
ptr = 0  # pointer into all_actual / all_predictions

for i, fr in enumerate(fold_results):
    n = fr["n_test"]
    act  = all_actual[ptr: ptr + n]
    pred = all_predictions[ptr: ptr + n]
    ptr += n

    ax = axes[i]
    ax.scatter(act, pred, alpha=0.6, color=fold_colors[i % len(fold_colors)], s=20)
    lim = [min(act + pred) - 5, max(act + pred) + 5]
    ax.plot(lim, lim, "k--", linewidth=1, label="Perfect")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Actual RUL")
    ax.set_ylabel("Predicted RUL")
    ax.set_title(
        f"Fold {fr['fold']}: Test={fr['test_battery']}\n"
        f"MAE={fr['mae']:.2f}  R²={fr['r2']:.3f}",
        fontsize=10
    )
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Actual vs Predicted RUL — LOGO Folds", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Per-fold metric bar chart
metrics = ["mae", "rmse", "r2"]
labels  = ["MAE (cycles)", "RMSE (cycles)", "R²"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, metric, label in zip(axes, metrics, labels):
    bars = ax.bar(
        results_df["test_battery"],
        results_df[metric],
        color=fold_colors[:len(results_df)],
        width=0.5,
        edgecolor="white",
    )
    ax.set_xlabel("Test Battery")
    ax.set_ylabel(label)
    ax.set_title(label, fontweight="bold")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Per-Fold XGBoost RUL Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (last fold's model)
importances = pd.Series(
    model.feature_importances_,
    index=FEATURE_COLUMNS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importances.index, importances.values,
               color="#4C72B0", edgecolor="white")
ax.set_xlabel("Feature Importance (XGBoost)", fontweight="bold")
ax.set_title("Feature Importance — Last Fold", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Summary

| Item | Value |
|------|-------|
| Model | XGBoost Regressor |
| Validation | Leave-One-Battery-Out (LOGO) |
| Training batteries | B0005, B0006, B0018 |
| Censored (excluded) | B0007 (EOL never reached) |
| Features used | 15 (frozen Phase 1) |
| EOL threshold | SOH ≤ 70% |

### B0007 Documentation
B0007's SOH never drops to 70% within the recorded cycles.  
Per the ML contract, it is treated as a **censored observation**:  
`rul_valid = False` for all rows — excluded from training and evaluation.  
It cannot be used as a LOGO fold test battery for RUL prediction.

In [ ]:
# Final summary
print("=" * 50)
print("  Phase 2 RUL — Final Summary")
print("=" * 50)
print(f"  Valid training rows : {len(X)}")
print(f"  Features            : {len(FEATURE_COLUMNS)}")
print(f"  LOGO folds          : {n_splits}")
print(f"  Overall MAE         : {overall_mae:.4f} cycles")
print(f"  Overall RMSE        : {overall_rmse:.4f} cycles")
print(f"  Overall R²          : {overall_r2:.4f}")
print("=" * 50)